In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
pip install openai anthropic google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.5/357.5 kB 7.9 MB/s eta 0:00:00ta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
OPENAI_API_KEY=""

In [7]:
# --- Setup (edit as needed) ---
import os, json, datetime, pathlib
PROVIDER = "openai"  # or "anthropic" / "gemini"
MODEL = "gpt-4o-mini"
GEN_ARGS = dict(temperature=0.2)
OUT_DIR = pathlib.Path("week1_runs"); OUT_DIR.mkdir(exist_ok=True)
def log_result(kind, name, prompt, output):
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    (OUT_DIR / f"{kind}_{name}_{ts}.prompt.txt").write_text(prompt, encoding="utf-8")
    (OUT_DIR / f"{kind}_{name}_{ts}.output.txt").write_text(output, encoding="utf-8")
    print(f"Logged → {OUT_DIR}")

In [11]:
# --- Minimal OpenAI client helper ---
from openai import OpenAI
_openai = OpenAI(api_key=OPENAI_API_KEY)
    # api_key=os.getenv(OPENAI_API_KEY))
def llm_openai(prompt, model=MODEL, **gen_args):
    resp = _openai.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":prompt}],
        **gen_args
    )
    return resp.choices[0].message.content


In [12]:
# --- Generic runner (wire your provider here) ---
def run(prompt, provider=PROVIDER, **gen_args):
    if provider == "openai":
        return llm_openai(prompt, **gen_args)
    raise NotImplementedError("Add anthropic/gemini helpers if needed.")

1) Zero‑Shot Prompting
How: Give clear instructions without examples.

When: Use for simple, unambiguous tasks (translations, factual queries).

In [13]:
# Run: zero-shot
PROMPT = r"""Translate the English phrase 'Flowers on the road' to Spanish."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("zero-shot", "example", PROMPT, out)


The phrase 'Flowers on the road' translates to Spanish as 'Flores en el camino'.
Logged → week1_runs


How: Give clear instructions and add a single example to demonstrate the task/output.

When: Use when one example is enough to clarify the task or format.

Example prompt

In uppercase, return the Spanish translation of the English word 'basket' only.
EXAMPLE:
English word (input): River
Spanish translation (output): RÍO

In [14]:
# Run: one-shot
PROMPT = r"""In uppercase, return the Spanish translation of the English word 'basket' only.
EXAMPLE:
English word (input): River
Spanish translation (output): RÍO"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("one-shot", "example", PROMPT, out)

CESTA
Logged → week1_runs


How: Give clear instructions with a few examples in the prompt.

When: Use to adapt the model to a task/domain without fine‑tuning; improves consistency.

Example prompt

Return the sentiment of this statement 'The lecture was quite boring' only. Either Positive, Negative, or Neutral.
EXAMPLES:
'This movie was great!': Positive
'I hated the service.': Negative
'I don't know how I feel about it.': Neutral

In [15]:
# Run: few-shot
PROMPT = r"""Return the sentiment of this statement 'The lecture was quite boring' only. Either Positive, Negative, or Neutral.
EXAMPLES:
'This movie was great!': Positive
'I hated the service.': Negative
'I don't know how I feel about it.': Neutral"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("few-shot", "example", PROMPT, out)


Negative
Logged → week1_runs


4) Role Prompting
How: Give clear instructions and assign a persona to steer tone and perspective.

When: Use for open‑ended tasks where tone or viewpoint matters.
Example prompt

ROLE: Act as a sweet college girl who uses Gen‑Z slang.
TASK: Write a short blog (≈500 words) with 4 points about college hacks.

In [16]:
# Run: role
PROMPT = r"""ROLE: Act as a sweet college girl who uses Gen‑Z slang.
TASK: Write a short blog (≈500 words) with 4 points about college hacks."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("role", "example", PROMPT, out)


**Title: College Hacks That’ll Make You Say “Bet!”**

Hey besties! 🌸✨ So, college can be a whole vibe, but let’s be real—it can also be super overwhelming. Between classes, assignments, and trying to maintain a social life, it’s easy to feel like you’re drowning in responsibilities. But don’t worry, I gotchu! Here are four college hacks that’ll totally help you slay the semester and keep your mental health in check. Let’s get into it! 💖

### 1. **Use Your Planner Like a Boss 🗓️**

Okay, so I know everyone says this, but hear me out! A planner is literally your best friend in college. Whether you go for a cute physical planner or a digital one (shoutout to Google Calendar!), keeping track of your assignments, exams, and social events is a game-changer. I like to color-code mine—like, pink for assignments, blue for exams, and green for fun stuff. It makes everything look super aesthetic and helps me stay organized. Plus, there’s something so satisfying about checking things off your list

5) Style Prompting
How: Specify desired style, tone, or genre explicitly.

When: Use when the output must match a particular style or tone.

Example prompt

Write a brief formal email requesting a raise.

In [17]:
# Run: style
PROMPT = r"""Write a brief formal email requesting a raise."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("style", "example", PROMPT, out)

Subject: Request for Salary Review

Dear [Manager's Name],

I hope this message finds you well. I am writing to formally request a review of my current salary. Over the past [duration, e.g., year], I have taken on additional responsibilities and contributed to [specific projects or achievements], which I believe have positively impacted our team's performance and the company's goals.

I would appreciate the opportunity to discuss my contributions and the possibility of a salary adjustment that reflects my efforts and the value I bring to the team.

Thank you for considering my request. I look forward to your response.

Best regards,

[Your Name]  
[Your Position]  
[Your Contact Information]  
Logged → week1_runs


6) Emotion Prompting
How: Add an emotionally charged phrase/sentence to shape affect.

When: Use in creative writing (stories, poetry) where emotion matters.

Example prompt

Write a poem about my lost imaginary friend who never gave up. I still miss my friend.

In [18]:
# Run: emotion
PROMPT = r"""Write a poem about my lost imaginary friend who never gave up. I still miss my friend."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("emotion", "example", PROMPT, out)

In the quiet corners of my mind,  
Where shadows dance and dreams unwind,  
There lived a friend, both brave and true,  
A spark of joy, a shade of blue.  

With laughter woven in the air,  
We spun our tales, a world to share,  
Through fields of wonder, skies so wide,  
Together we would laugh and glide.  

You wore a cloak of starlit night,  
With eyes that shimmered, pure delight,  
In every secret, every game,  
You whispered softly, called my name.  

But time, that thief, with gentle hands,  
Unraveled dreams like shifting sands,  
And though you lingered, strong and bright,  
The world grew dimmer, lost its light.  

Yet still I feel your spirit near,  
In echoes soft, in whispered cheer,  
You taught me strength, to never yield,  
A heart unbroken, a spirit healed.  

Though paths have parted, skies grown gray,  
I hold your memory, come what may,  
For in my heart, you’ll always stay,  
My steadfast friend, my light, my way.  

So here’s to you, my brave old soul,  
In every 

7) Contextual Prompting
How: Provide background/custom info before instructions.

When: Use when domain‑specific context is needed (RAG/chatbots).

Example prompt

CONTEXT: My name is Jennifer Luke and I'm a marketing manager at JL Firm.
Write an email to the team about the upcoming campaign.

In [19]:
# Run: contextual
PROMPT = r"""CONTEXT: My name is Jennifer Luke and I'm a marketing manager at JL Firm.
Write an email to the team about the upcoming campaign."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("contextual", "example", PROMPT, out)


Subject: Exciting Upcoming Campaign – Let’s Get Started!

Hi Team,

I hope this message finds you well! I’m thrilled to announce that we are gearing up for our upcoming marketing campaign, and I wanted to share some key details and next steps with all of you.

**Campaign Overview:**
Our new campaign will focus on [briefly describe the campaign theme or goal, e.g., promoting our latest product, increasing brand awareness, etc.]. We aim to launch on [insert launch date], and I believe this initiative has the potential to make a significant impact.

**Key Objectives:**
- [Objective 1]
- [Objective 2]
- [Objective 3]

**Next Steps:**
1. **Kick-off Meeting:** We will hold a kick-off meeting on [insert date and time] to discuss the campaign in detail, brainstorm ideas, and assign roles. Please make it a priority to attend.
2. **Research & Insights:** I encourage everyone to gather insights and data relevant to our target audience. This will help us tailor our messaging effectively.
3. **Crea

8) Rephrase‑and‑Respond (RaR)
How: Tell the model to rewrite the question as a better prompt, then answer it.

When: Use for complex tasks to improve understanding and accuracy.

Example prompt

Rephrase and expand the following question, then answer it: What is the difference between correlation and causation?

In [20]:
# Run: rephrase-and-respond
PROMPT = r"""Rephrase and expand the following question, then answer it: What is the difference between correlation and causation?"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("rephrase-and-respond", "example", PROMPT, out)


**Rephrased and Expanded Question:**
What distinguishes correlation from causation in statistical analysis, and why is it important to understand the difference between these two concepts when interpreting data?

**Answer:**
Correlation and causation are two fundamental concepts in statistics that are often confused, yet they represent very different relationships between variables.

**Correlation** refers to a statistical measure that indicates the extent to which two variables fluctuate together. When two variables are correlated, it means that as one variable changes, the other variable tends to change in a specific direction—either increasing or decreasing. Correlation is quantified using a correlation coefficient, which ranges from -1 to 1. A coefficient close to 1 indicates a strong positive correlation, while a coefficient close to -1 indicates a strong negative correlation. However, correlation does not imply that one variable causes the other to change.

**Causation**, on the 

9) Re‑Reading (RE2)
How: Ask the question, add 'Read that again:' and repeat the question verbatim.

When: Use for reasoning problems that benefit from refocus.

Example prompt

A farmer has a rectangular field that is 3× as long as it is wide. The perimeter is 400 m. What are the dimensions?
Read that again: "A farmer has a rectangular field that is 3× as long as it is wide. The perimeter is 400 m. What are the dimensions?"

In [23]:
# Run: re-reading
PROMPT = r"""A farmer has a rectangular field that is 3× as long as it is wide. The perimeter is 400 m. What are the dimensions?
Read that again: "A farmer has a rectangular field that is 3× as long as it is wide. The perimeter is 400 m. What are the dimensions?"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("re-reading", "example", PROMPT, out)


Let the width of the rectangular field be \( w \) meters. According to the problem, the length \( l \) is 3 times the width, so we can express the length as:

\[
l = 3w
\]

The formula for the perimeter \( P \) of a rectangle is given by:

\[
P = 2l + 2w
\]

We know the perimeter is 400 meters, so we can set up the equation:

\[
2l + 2w = 400
\]

Substituting \( l = 3w \) into the perimeter equation:

\[
2(3w) + 2w = 400
\]

This simplifies to:

\[
6w + 2w = 400
\]

Combining like terms gives:

\[
8w = 400
\]

Now, solving for \( w \):

\[
w = \frac{400}{8} = 50 \text{ meters}
\]

Now that we have the width, we can find the length:

\[
l = 3w = 3 \times 50 = 150 \text{ meters}
\]

Thus, the dimensions of the rectangular field are:

- Width: \( 50 \) meters
- Length: \( 150 \) meters
Logged → week1_runs


10) System Prompting
How: Set high‑level behavior/tone in a system message (or app system prompt).

When: Use to govern the assistant across a conversation.

Example prompt

SYSTEM: You are a helpful assistant that provides factual, concise responses.

In [24]:
# Run: system
PROMPT = r"""SYSTEM: You are a helpful assistant that provides factual, concise responses."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("system", "example", PROMPT, out)

Understood! How can I assist you today?
Logged → week1_runs


11) Self‑Ask
How: Tell the model to decompose the question into sub‑questions and answer each.

When: Use for multi‑step reasoning and decision‑making tasks.

Example prompt

Should I pursue a master's in data science? Break this into sub‑questions, answer them, then provide a final recommendation.

In [25]:
# Run: self-ask
PROMPT = r"""Should I pursue a master's in data science? Break this into sub‑questions, answer them, then provide a final recommendation."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("self-ask", "example", PROMPT, out)


Certainly! Let's break down the decision to pursue a master's in data science into several sub-questions:

### 1. What are my career goals?
**Answer:** Consider what you want to achieve in your career. If you aim to work in data-intensive fields such as machine learning, artificial intelligence, or big data analytics, a master's in data science can provide you with the necessary skills and credentials. If your goals are more aligned with roles that do not require advanced statistical or programming knowledge, a master's may not be essential.

### 2. Do I have a background in relevant fields?
**Answer:** Evaluate your current educational background and work experience. A strong foundation in mathematics, statistics, and programming (especially in languages like Python or R) is beneficial. If you lack this background, a master's program can help bridge that gap. If you already have experience in data-related roles, you might consider whether a master's is necessary for your advancement.


12) Chain‑of‑Thought (CoT)
How: Add a reasoning nudge such as 'Let's think step by step.'

When: Use for math/logic problems. Prefer brief step outlines over exposing hidden chain‑of‑thought in production.

Example prompt

What is the total cost of a meal with a 10% discount and 7% tax? Let's think step by step.

In [26]:
# Run: chain-of-thought
PROMPT = r"""What is the total cost of a meal with a 10% discount and 7% tax? Let's think step by step."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("chain-of-thought", "example", PROMPT, out)


To calculate the total cost of a meal with a 10% discount and a 7% tax, we can follow these steps:

1. **Determine the original price of the meal.** Let's denote the original price as \( P \).

2. **Calculate the discount.** The discount is 10% of the original price:
   \[
   \text{Discount} = 0.10 \times P
   \]

3. **Calculate the price after the discount.** Subtract the discount from the original price:
   \[
   \text{Price after discount} = P - \text{Discount} = P - (0.10 \times P) = 0.90 \times P
   \]

4. **Calculate the tax on the discounted price.** The tax is 7% of the price after the discount:
   \[
   \text{Tax} = 0.07 \times (0.90 \times P) = 0.063 \times P
   \]

5. **Calculate the total cost.** Add the tax to the price after the discount:
   \[
   \text{Total cost} = \text{Price after discount} + \text{Tax} = (0.90 \times P) + (0.063 \times P) = (0.90 + 0.063) \times P = 0.963 \times P
   \]

Thus, the total cost of the meal after applying the discount and adding the tax 

13) Step‑Back Prompting
How: Ask a broad question, then a targeted one based on the broader answer.

When: Use for analysis/decisions that depend on multiple factors.

Example prompt

Explain key factors that influence expansion to a new market. Based on this, should a tech company expand to Europe?

In [27]:
# Run: step-back
PROMPT = r"""Explain key factors that influence expansion to a new market. Based on this, should a tech company expand to Europe?"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("step-back", "example", PROMPT, out)


Expanding to a new market is a complex decision that involves several key factors. Here are some of the most important considerations:

### Key Factors Influencing Market Expansion

1. **Market Research and Demand**:
   - **Consumer Behavior**: Understanding local preferences, buying habits, and cultural nuances is crucial.
   - **Market Size and Growth Potential**: Assessing the size of the target market and its growth trajectory can indicate potential profitability.

2. **Regulatory Environment**:
   - **Legal and Compliance Issues**: Different countries have varying regulations regarding data protection, labor laws, and business operations.
   - **Trade Policies and Tariffs**: Understanding import/export regulations and any tariffs that may apply is essential.

3. **Competitive Landscape**:
   - **Existing Competitors**: Analyzing the competition can provide insights into market saturation and potential barriers to entry.
   - **Market Positioning**: Identifying how to differentiate

14) Self‑Consistency
How: Ask for multiple candidate answers; return the most frequent.

When: Use when many answers are possible and you want higher reliability.

Example prompt

What is the most popular programming language for machine learning? Generate 5 answers and return the one that appears most often.

In [31]:
# Run: self-consistency
PROMPT = r"""What is the most popular programming language for machine learning? Generate 5 answers and return the one that appears most often."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("self-consistency", "example", PROMPT, out)

The most popular programming languages for machine learning are:

1. Python
2. R
3. Java
4. C++
5. Julia

Among these, **Python** is the most popular programming language for machine learning, as it is widely used due to its simplicity, extensive libraries (like TensorFlow, PyTorch, and scikit-learn), and strong community support.
Logged → week1_runs


15) Thread‑of‑Thought (ThoT)
How: Ask to 'walk through in manageable parts step by step' (briefly).

When: Use for Q&A over large, complex contexts (e.g., RAG).

Example prompt

CONTEXT: Party music preferences problem (10 guests, 3 genres, limited speakers).
Walk me through in manageable parts to maximize satisfied guests.

In [30]:
# Run: thread-of-thought
PROMPT = r"""CONTEXT: Party music preferences problem (10 guests, 3 genres, limited speakers).
Walk me through in manageable parts to maximize satisfied guests."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("thread-of-thought", "example", PROMPT, out)

To maximize guest satisfaction with their music preferences at a party with 10 guests, 3 genres, and limited speakers, we can break the problem down into manageable parts. Here’s a step-by-step approach:

### Step 1: Gather Information
1. **Identify Guests' Preferences**: Create a list of the 10 guests and ask each one to rank their preferences for the 3 genres (e.g., Pop, Rock, Hip-Hop). You can use a simple ranking system (1-3) where 1 is their favorite genre.
  
2. **Count Preferences**: Tally the number of guests who prefer each genre. This will help you understand which genres are most popular among your guests.

### Step 2: Analyze the Data
1. **Determine Majority Preferences**: Identify which genre has the highest preference count. This will be your primary genre for the party.

2. **Identify Secondary Preferences**: Look at the second and third most popular genres. This will help you decide how to allocate time for each genre during the party.

### Step 3: Plan the Music Schedu

16) Tree‑of‑Thought (ToT)
How: At each step, generate multiple options, evaluate them, pick the best, continue.

When: Use for deep reasoning, multi‑step planning, high accuracy.

Example prompt

Design a coffee cup that keeps drinks hot longer. Break down steps; at each, generate options, evaluate (feasibility/cost/impact), choose best, iterate to a final design.

In [29]:
# Run: tree-of-thought
PROMPT = r"""Design a coffee cup that keeps drinks hot longer. Break down steps; at each, generate options, evaluate (feasibility/cost/impact), choose best, iterate to a final design."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("tree-of-thought", "example", PROMPT, out)

Designing a coffee cup that keeps drinks hot longer involves several steps, including material selection, insulation methods, shape design, and additional features. Below is a structured breakdown of the design process, including options, evaluations, and iterations.

### Step 1: Material Selection

**Options:**
1. **Stainless Steel**: Durable, good thermal conductivity.
2. **Ceramic**: Good heat retention, aesthetically pleasing.
3. **Glass**: Non-reactive, can be double-walled for insulation.
4. **Plastic (BPA-free)**: Lightweight, cost-effective but less effective at retaining heat.

**Evaluation:**
- **Stainless Steel**: High durability and excellent heat retention. Moderate cost.
- **Ceramic**: Good heat retention but can be fragile. Moderate cost.
- **Glass**: Aesthetic appeal but can be heavy and fragile. Higher cost for double-walled.
- **Plastic**: Low cost but poor heat retention.

**Best Choice**: **Stainless Steel** for its durability and heat retention properties.

### Ste

17) ReAct (Reason + Act)
How: Alternate between thought, action (tool call), and observation; refine iteratively.

When: Use when interacting with external tools/APIs in loops.

Example prompt

Find the latest EV market trends. First list keywords, then call a search API, observe results, refine, and repeat until most relevant, recent data is found.

In [28]:
# Run: react
PROMPT = r"""Find the latest EV market trends. First list keywords, then call a search API, observe results, refine, and repeat until most relevant, recent data is found."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("react", "example", PROMPT, out)


I don't have the capability to call external APIs or perform live searches. However, I can help you identify relevant keywords and suggest a strategy for finding the latest EV market trends. Here’s how you can proceed:

### Step 1: Identify Keywords
Start with a list of keywords that are relevant to the electric vehicle (EV) market. Here are some suggestions:

1. Electric Vehicles (EV)
2. EV Market Trends
3. EV Sales Growth
4. EV Adoption Rates
5. EV Charging Infrastructure
6. Battery Technology
7. EV Manufacturers
8. Government Policies on EVs
9. Sustainable Transportation
10. EV Market Forecast 2023
11. Electric Vehicle Incentives
12. EV Consumer Preferences
13. Autonomous Electric Vehicles
14. EV Market Challenges
15. Global EV Market Analysis

### Step 2: Conduct a Search
Use a search engine or news aggregator to find the latest articles, reports, and data on the EV market. You can use the keywords above to refine your search. Here are some suggested search queries:

- "latest EV m

In [ ]:
# Optional: Validate JSON outputs for structured prompts
def ensure_json(text):
    try:
        return json.loads(text)
    except Exception:
        import re
        m = re.search(r"\{[\s\S]*\}", text)
        if m:
            return json.loads(m.group(0))
        raise